In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, precision_score, recall_score

# Data Loading and Initial Exploration

In [16]:
df = pd.read_csv("titanic_data_updated.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
68,69,yes,third,"Andersson, Miss. Erna Alexandra",female,17.0,4,2,3101281,7.9250,NaN,S
361,362,no,second,"del Carlo, Mr. Sebastiano",male,29.0,1,0,SC/PARIS 2167,27.7208,NaN,C
750,751,yes,second,"Wells, Miss. Joan",female,4.0,1,1,29103,23.0000,NaN,S
60,61,no,third,"Sirayanian, Mr. Orsen",male,22.0,0,0,2669,7.2292,NaN,C
258,259,yes,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C


In [17]:
df['Cabin'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


# Feature Engineering

In [18]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
367,368,yes,third,"Moussa, Mrs. (Mantoura Boulos)",female,NaN,0,0,2626,7.2292,Missing,C,1,M
339,340,no,first,"Blackwell, Mr. Stephen Weart",male,45.0,0,0,113784,35.5000,T,S,1,T
505,506,no,first,"Penasco y Castellana, Mr. Victor de Satode",male,18.0,1,0,PC 17758,108.9000,C65,C,2,C
438,439,no,first,"Fortune, Mr. Mark",male,64.0,1,4,19950,263.0000,C23 C25 C27,S,6,C
597,598,no,third,"Johnson, Mr. Alfred",male,49.0,0,0,LINE,0.0000,Missing,S,1,M


In [19]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [20]:
X = df.drop('Survived', axis=1)
y = df['Survived']

# Randrom state , Stratify and Train Test Split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Implementation of Preprocessor Pipeline

In [22]:
# Pipeline
# Numerical Values
p1 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]
)

In [23]:
categories = [['third', 'second', 'first']]

In [24]:
# Pipeline
# Categorical Column
p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=categories)),
        ('scaler', MinMaxScaler())
    ]
)

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipe_1', p1, ['Age']),
        ('pipe_2', p2, ['Fare', 'Family_Size']),
        ('pipe_3', p3, ['Embarked', 'Sex', 'Deck']),
        ('pipe_4', p4, ['Pclass']),
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipe_1', ...), ('pipe_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


# Label Encoding

In [26]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)

# Training the Model

In [27]:
sv_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', SVC())
    ]
)
sv_model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipe_1', ...), ('pipe_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
grid_pram = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.01, 0.1, 1, 10, 50, 100],
    },
    {
        "model__kernel": ["rbf"],
        "model__C": [0.01, 0.1, 1, 10, 50, 100],
        "model__gamma": [0.01, 0.1, 1, 5, 10, "scale", "auto"],
    },
    {
        "model__kernel": ["poly"],
        "model__C": [0.01, 0.1, 1, 10, 50, 100],
        "model__degree": [2, 3],
    },
]

best_sv_model = GridSearchCV(
    estimator=sv_model, 
    param_grid=grid_pram, 
    cv=5
)

best_sv_model.fit(X_train, y_train)

In [ ]:
best_sv_model.fit(X_train, y_train)

In [ ]:
y_pred_train = best_sv_modeles.predict(X_train)

print(accuracy_score(y_train, y_pred_train))

In [ ]:
y_pred = best_sv_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)
precision = precision_score(y_test, y_pred)
print(precision)
recall = recall_score(y_test, y_pred)
print(recall)